# Clinical + Dietary + Anthro GPU/CPU Calibrated Pipeline

This notebook builds a detailed end-to-end pipeline on the multi-year `Datasets` folder using only Clinical, Dietary, and Anthro sources.

Included stages:
1. Load raw datasets from `Datasets/Clinical`, `Datasets/Dietary`, and `Datasets/Anthro`.
2. Left-join Dietary and Anthro onto Clinical using the strongest common clinical keys.
3. Reuse the repo's manual variable dropping, anthropometric filtering, collinearity thresholding, scaling, and KNN imputation pattern.
4. Tune GPU models more rigorously than CPU models.
5. Train lighter CPU baselines.
6. Calibrate the strongest models at the end and evaluate the best calibrated setup on held-out test data.

In [1]:
%pip install -q numpy pandas scipy scikit-learn xgboost catboost lightgbm joblib venn-abers

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import subprocess
import time
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.ensemble import AdaBoostClassifier
from sklearn.impute import KNNImputer
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, log_loss, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import ParameterGrid, train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier, XGBRFClassifier

warnings.filterwarnings('ignore')

RANDOM_SEED = 2026
np.random.seed(RANDOM_SEED)
COLLINEARITY_THRESHOLD = 0.70
TEST_SIZE = 0.15
CAL_SIZE = 0.15

def gpu_available() -> bool:
    try:
        subprocess.check_output(['nvidia-smi'], stderr=subprocess.STDOUT, text=True)
        return True
    except Exception:
        return False

GPU_AVAILABLE = gpu_available()
print(f'Random seed: {RANDOM_SEED}')
print(f'GPU available: {GPU_AVAILABLE}')
print('xgboost version:', __import__('xgboost').__version__)
print('catboost version:', __import__('catboost').__version__)
print('lightgbm version:', __import__('lightgbm').__version__)

Random seed: 2026
GPU available: True
xgboost version: 3.2.0
catboost version: 1.2.10
lightgbm version: 4.6.0


In [3]:
PROJECT_ROOT_CANDIDATES = [Path.cwd(), Path.cwd().parent, Path('/workspace/Thesis-part-2')]
PROJECT_ROOT = next((p for p in PROJECT_ROOT_CANDIDATES if (p / 'Datasets').exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate project root containing 'Datasets'.")

DATASETS_DIR = PROJECT_ROOT / 'Datasets'
CLINICAL_DIR = DATASETS_DIR / 'Clinical'
DIETARY_DIR = DATASETS_DIR / 'Dietary'
ANTHRO_DIR = DATASETS_DIR / 'Anthro'

ARTIFACT_DIR = PROJECT_ROOT / 'V2.2.1.1' / 'clinical_dietary_anthro_gpu_cpu_artifacts'
MODEL_DIR = ARTIFACT_DIR / 'models'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

def find_dataset_file(folder: Path, dataset_hint: str, extra_tokens=()):
    if not folder.exists():
        raise FileNotFoundError(f'Folder not found: {folder}')

    def _matches(path: Path) -> bool:
        name = path.name.lower()
        if 'dictionary' in name:
            return False
        if 'data-set' in name and dataset_hint in name:
            return True
        return all(token in name for token in extra_tokens) and 'data-set' in name

    local_candidates = sorted([p for p in folder.glob('*.csv') if _matches(p)])
    if local_candidates:
        return local_candidates[0]

    fallback_candidates = sorted([p for p in PROJECT_ROOT.rglob('*.csv') if _matches(p)])
    if fallback_candidates:
        chosen = fallback_candidates[0]
        print(f'Warning: using fallback file outside expected folder for {dataset_hint}: {chosen}')
        return chosen

    raise FileNotFoundError(
        f'No usable data-set CSV found for {dataset_hint}. Checked {folder} and recursive fallbacks under {PROJECT_ROOT}. '
        'Current repo snapshot may be missing the raw dataset file.'
    )

clinical_path = find_dataset_file(CLINICAL_DIR, 'clinical')
dietary_path = find_dataset_file(DIETARY_DIR, 'dietary', extra_tokens=('dietary',))
anthro_path = find_dataset_file(ANTHRO_DIR, 'anthrop', extra_tokens=('anthrop',))

clinical_df = pd.read_csv(clinical_path, low_memory=False)
dietary_df = pd.read_csv(dietary_path, low_memory=False)
anthro_df = pd.read_csv(anthro_path, low_memory=False)

print(f'Clinical file: {clinical_path.name} -> {clinical_df.shape}')
print(f'Dietary file: {dietary_path.name} -> {dietary_df.shape}')
print(f'Anthro file: {anthro_path.name} -> {anthro_df.shape}')
print(f'Artifacts directory: {ARTIFACT_DIR}')

Clinical file: Jonathan Ralph_Baes_2025-09-18020631_data-set_clinical.csv -> (351701, 29)
Dietary file: Jonathan Ralph_Baes_2026-03-26141801_data-set_dietary.csv -> (9925, 45)
Anthro file: Jonathan Ralph_Baes_2025-09-29103911_data-set_anthrop.csv -> (444400, 18)
Artifacts directory: /workspace/Thesis-part-2/V2.2.1.1/clinical_dietary_anthro_gpu_cpu_artifacts


In [4]:
merge_key_options = [
    ['enns_year', 'hhnum', 'member_code'],
    ['hhnum', 'member_code'],
    ['enns_year', 'hhnum'],
    ['hhnum'],
]

def strongest_common_keys(base_df: pd.DataFrame, other_df: pd.DataFrame):
    return next((keys for keys in merge_key_options if all(k in base_df.columns and k in other_df.columns for k in keys)), [])

def left_merge_on_clinical(base_df: pd.DataFrame, right_df: pd.DataFrame, label: str):
    keys = strongest_common_keys(base_df, right_df)
    if not keys:
        common = sorted(set(base_df.columns).intersection(set(right_df.columns)))
        raise ValueError(f'No usable merge keys found for {label}. Common columns: {common}')

    work = right_df.copy()
    dup_count = int(work.duplicated(subset=keys).sum())
    if dup_count > 0:
        print(f'Warning: {label} has {dup_count:,} duplicate rows on {keys}; keeping first per key.')
        work = work.sort_values(by=keys).drop_duplicates(subset=keys, keep='first')

    overlap_non_keys = [c for c in work.columns if c in base_df.columns and c not in keys]
    work = work.drop(columns=overlap_non_keys, errors='ignore')
    merged = base_df.merge(work, on=keys, how='left')

    info = {
        'source': label,
        'merge_keys': keys,
        'dropped_overlap_non_keys': len(overlap_non_keys),
        'rows_preserved': len(merged) == len(base_df),
        'merged_shape': merged.shape,
    }
    print(info)
    return merged, info

merged_df, dietary_merge_info = left_merge_on_clinical(clinical_df, dietary_df, 'dietary')
merged_df, anthro_merge_info = left_merge_on_clinical(merged_df, anthro_df, 'anthro')

merged_out = ARTIFACT_DIR / 'merged_clinical_dietary_anthro_multiyear.csv'
merged_df.to_csv(merged_out, index=False)
print(f'Merged dataset saved: {merged_out}')
print(f'Final merged shape: {merged_df.shape}')

{'source': 'dietary', 'merge_keys': ['hhnum'], 'dropped_overlap_non_keys': 1, 'rows_preserved': True, 'merged_shape': (351701, 72)}
{'source': 'anthro', 'merge_keys': ['enns_year', 'hhnum', 'member_code'], 'dropped_overlap_non_keys': 7, 'rows_preserved': True, 'merged_shape': (351701, 80)}
Merged dataset saved: /workspace/Thesis-part-2/V2.2.1.1/clinical_dietary_anthro_gpu_cpu_artifacts/merged_clinical_dietary_anthro_multiyear.csv
Final merged shape: (351701, 80)


In [5]:
def find_bp_column(df: pd.DataFrame, kind: str) -> str:
    preferred = {
        'sbp': ['Ave_SBP', 'ave_sbp', 'SBP', 'sbp'],
        'dbp': ['Ave_DBP', 'ave_dbp', 'DBP', 'dbp'],
    }
    for col in preferred[kind]:
        if col in df.columns:
            return col
    probe = [c for c in df.columns if kind.upper() in c.upper()]
    if probe:
        return probe[0]
    raise KeyError(f'Could not find {kind.upper()} column in merged data.')

def map_alcohol_level(row):
    status = row.get('alcohol_status', np.nan)
    binge = row.get('binge_drink', row.get('binge_drinking', np.nan))
    if pd.isna(status):
        return np.nan
    if status == 0:
        return 0
    if status == 2:
        return 1
    if status == 1:
        return 3 if binge == 1 else 2
    return np.nan

def map_smoking_level(row):
    status = row.get('smoke_status', np.nan)
    current = row.get('currentsmoking', row.get('current_smoking', np.nan))
    if pd.isna(status):
        return np.nan
    if status == 0:
        return 0
    if status == 2:
        return 1
    if status == 1:
        return 3 if current == 3 else 2
    return np.nan

model_df = merged_df.copy()
sbp_col = find_bp_column(model_df, 'sbp')
dbp_col = find_bp_column(model_df, 'dbp')
model_df['Hypertension'] = ((model_df[sbp_col] >= 130) | (model_df[dbp_col] >= 80)).astype(int)

if {'height', 'weight'}.issubset(model_df.columns):
    model_df['BMI'] = model_df['weight'] / ((model_df['height'] / 100.0) ** 2)

special_na_codes = {
    'ever_smk': [99],
    'con_alcohol': [9],
    'drnk_30days': [9],
    'currentsmoking': [99],
    'current_smoking': [99],
    'binge_drink': [9, 99],
    'binge_drinking': [9, 99],
}
for col, codes in special_na_codes.items():
    if col in model_df.columns:
        model_df[col] = model_df[col].replace(codes, np.nan)

if 'alcohol_status' in model_df.columns:
    model_df['alcohol_level'] = model_df.apply(map_alcohol_level, axis=1)
if 'smoke_status' in model_df.columns:
    model_df['smoking_level'] = model_df.apply(map_smoking_level, axis=1)

if 'anthro_group' in model_df.columns:
    before_rows = len(model_df)
    model_df = model_df[~model_df['anthro_group'].isin([5, 6])].copy()
    print(f'Excluded anthro_group 5 and 6: {before_rows - len(model_df):,} rows removed')

manual_drop_candidates = [
    sbp_col, dbp_col,
    'height', 'weight',
    'enns_year', 'hhnum', 'member_code', 'anthro_group',
    'regcode', 'provhuc', 'provcode', 'psc', 'psurec', 'csc', 'rhc', 'strrec',
    'ms_psucode', 'rep_natl', 'rep_prov', 'wrkplace', 'interview_status', 'intdate', 'enumcode',
    'fwgth_natl_var', 'fwgth_prov', 'fwgth_natl2_var',
    'fwgti_natl_var', 'fwgti_prov', 'fwgti_natl2_var', 'fwgti_prov2',
    'finalwgt1', 'finalwgt4', 'wgts', 'cu',
    'mos_lactation', 'mos_preg', 'haswork', 'occup', 'educ',
    'alcohol', 'con_alcohol', 'drnk_30days', 'drnk_30d_num', 'alcohol_status', 'binge_drink', 'binge_drinking',
    'currentsmoking', 'current_smoking', 'ever_smk', 'smoke_status',
    'fg1', 'fg2', 'fg3', 'fg4', 'fg5', 'fg6', 'fg7', 'fg8', 'fg9', 'fg10',
    'fg11', 'fg12', 'fg13', 'fg14', 'fg15', 'fg16', 'fg17', 'fg18', 'fg19', 'fg20',
    'fg21', 'fg22', 'fg23', 'fg24', 'fg25', 'fg26', 'fg27',
    'Total_Food', 'Total_FoodIntake',
]
manual_existing_drops = [c for c in manual_drop_candidates if c in model_df.columns]
model_df = model_df.drop(columns=manual_existing_drops, errors='ignore')

drop_report = pd.DataFrame({
    'manual_drop_column': manual_existing_drops
})
drop_report.to_csv(ARTIFACT_DIR / 'manual_drop_report.csv', index=False)

print(f'Target created from: {sbp_col}, {dbp_col}')
print(f'Manual drops applied: {len(manual_existing_drops)}')
print(f'Model dataframe shape after manual drops: {model_df.shape}')
print(model_df['Hypertension'].value_counts(normalize=True).rename('ratio').to_string())

Excluded anthro_group 5 and 6: 14,526 rows removed
Target created from: Ave_SBP, Ave_DBP
Manual drops applied: 60
Model dataframe shape after manual drops: (337175, 24)
Hypertension
0    0.666755
1    0.333245


In [6]:
numeric_for_corr = model_df.select_dtypes(include=[np.number]).copy()
exclude_corr_cols = [c for c in ['Hypertension', 'enns_year', 'hhnum', 'member_code'] if c in numeric_for_corr.columns]
numeric_for_corr = numeric_for_corr.drop(columns=exclude_corr_cols, errors='ignore')
numeric_for_corr = numeric_for_corr.dropna(axis=0)

if numeric_for_corr.empty or numeric_for_corr.shape[1] < 2:
    corr = pd.DataFrame()
    high_corr_df = pd.DataFrame(columns=['feature_1', 'feature_2', 'correlation', 'abs_correlation'])
    auto_corr_drop_cols = []
else:
    corr = numeric_for_corr.corr()
    pairs = []
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    missing_rate = model_df.isna().mean().to_dict()
    keep_preference = {'BMI', 'ldl', 'age', 'sex', 'uic', 'vita', 'hemoglobin'}
    auto_corr_drop_cols = []

    for col in upper.columns:
        for row in upper.index:
            val = upper.loc[row, col]
            if pd.notna(val) and abs(val) >= COLLINEARITY_THRESHOLD:
                pairs.append({
                    'feature_1': row,
                    'feature_2': col,
                    'correlation': float(val),
                    'abs_correlation': float(abs(val)),
                })
                if row in keep_preference and col not in keep_preference:
                    drop_col = col
                elif col in keep_preference and row not in keep_preference:
                    drop_col = row
                else:
                    drop_col = row if missing_rate.get(row, 0.0) > missing_rate.get(col, 0.0) else col
                auto_corr_drop_cols.append(drop_col)

    high_corr_df = pd.DataFrame(pairs).sort_values('abs_correlation', ascending=False) if pairs else pd.DataFrame(columns=['feature_1', 'feature_2', 'correlation', 'abs_correlation'])
    auto_corr_drop_cols = sorted(set(auto_corr_drop_cols))

manual_collinearity_aliases = [
    'Total_Food_epwt', 'Total_Ener', 'Total_Energy', 'Total_Prot', 'Total_Protein', 'Total_Fat', 'Total_Iron',
    'Total_CHO', 'Total_Nia', 'Total_Niacin', 'Total_Ribo', 'Total_Riboflavin',
    'epwt_fg25', 'epwt_fg13', 'epwt_fg10', 'epwt_fg2', 'epwt_fg19', 'epwt_fg16',
    'hip', 'waist', 'chol'
]
manual_collinearity_drops = [c for c in manual_collinearity_aliases if c in model_df.columns]
final_collinearity_drops = sorted(set(auto_corr_drop_cols).union(manual_collinearity_drops))

collinearity_report = {
    'threshold': COLLINEARITY_THRESHOLD,
    'n_high_corr_pairs': int(len(high_corr_df)),
    'manual_collinearity_drops': manual_collinearity_drops,
    'auto_collinearity_drops': auto_corr_drop_cols,
    'final_collinearity_drops': final_collinearity_drops,
}
with open(ARTIFACT_DIR / 'collinearity_report.json', 'w', encoding='utf-8') as f:
    json.dump(collinearity_report, f, indent=2)
high_corr_df.to_csv(ARTIFACT_DIR / 'collinearity_pairs.csv', index=False)

model_df = model_df.drop(columns=final_collinearity_drops, errors='ignore')

print(f'Collinearity threshold: {COLLINEARITY_THRESHOLD}')
print(f'High-correlation pairs found: {len(high_corr_df)}')
print(f'Columns dropped for collinearity: {len(final_collinearity_drops)}')
if len(high_corr_df):
    print(high_corr_df.head(15).to_string(index=False))
print(f'Shape after collinearity drops: {model_df.shape}')

Collinearity threshold: 0.7
High-correlation pairs found: 21
Columns dropped for collinearity: 13
       feature_1        feature_2  correlation  abs_correlation
    Total_Energy        Total_CHO     0.945034         0.945034
    Total_Niacin    Total_Protein     0.907719         0.907719
            chol              ldl     0.894850         0.894850
    Total_Energy    Total_Protein     0.869568         0.869568
           waist              BMI     0.855418         0.855418
           waist              hip     0.850447         0.850447
    Total_Energy     Total_Niacin     0.839703         0.839703
             hip              BMI     0.832595         0.832595
   Total_Calcium    Total_Protein     0.802006         0.802006
   Total_Protein Total_Riboflavin     0.780857         0.780857
   Total_Calcium Total_Riboflavin     0.764752         0.764752
    Total_Energy    Total_Calcium     0.759643         0.759643
   Total_Calcium     Total_Niacin     0.754194         0.754194
      

In [7]:
target_col = 'Hypertension'
X = model_df.drop(columns=[target_col]).copy()
y = model_df[target_col].astype(int).copy()

categorical_cols = X.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
X = pd.get_dummies(X, columns=categorical_cols, drop_first=False, dummy_na=True)
X = X.replace([np.inf, -np.inf], np.nan)

X_train_raw, X_temp_raw, y_train, y_temp = train_test_split(
    X, y, test_size=TEST_SIZE + CAL_SIZE, random_state=RANDOM_SEED, stratify=y
)
relative_test_size = TEST_SIZE / (TEST_SIZE + CAL_SIZE)
X_cal_raw, X_test_raw, y_cal, y_test = train_test_split(
    X_temp_raw, y_temp, test_size=relative_test_size, random_state=RANDOM_SEED, stratify=y_temp
)

numeric_cols = X_train_raw.select_dtypes(include=[np.number]).columns.tolist()
train_medians = X_train_raw[numeric_cols].median()
scaler = StandardScaler()

X_train_std = pd.DataFrame(
    scaler.fit_transform(X_train_raw[numeric_cols].fillna(train_medians)),
    columns=numeric_cols, index=X_train_raw.index
)
X_cal_std = pd.DataFrame(
    scaler.transform(X_cal_raw[numeric_cols].fillna(train_medians)),
    columns=numeric_cols, index=X_cal_raw.index
)
X_test_std = pd.DataFrame(
    scaler.transform(X_test_raw[numeric_cols].fillna(train_medians)),
    columns=numeric_cols, index=X_test_raw.index
)

X_train_std = X_train_std.mask(X_train_raw[numeric_cols].isna())
X_cal_std = X_cal_std.mask(X_cal_raw[numeric_cols].isna())
X_test_std = X_test_std.mask(X_test_raw[numeric_cols].isna())

def masked_knn_rmse(X_df: pd.DataFrame, k: int, sample_rows: int = 5000, mask_rate: float = 0.05) -> float:
    complete_rows = X_df.dropna(axis=0)
    if len(complete_rows) < 200:
        return np.inf
    sample = complete_rows.sample(min(sample_rows, len(complete_rows)), random_state=RANDOM_SEED).copy()
    arr = sample.values.astype(float)
    rng = np.random.default_rng(RANDOM_SEED + k)
    mask = rng.random(arr.shape) < mask_rate
    if mask.sum() == 0:
        return np.inf
    masked = arr.copy()
    masked[mask] = np.nan
    imputer = KNNImputer(n_neighbors=k)
    imputed = imputer.fit_transform(masked)
    return float(np.sqrt(np.mean((imputed[mask] - arr[mask]) ** 2)))

k_grid = [3, 5, 7, 9, 11]
k_rows = []
for k in k_grid:
    k_rows.append({'k': k, 'rmse': masked_knn_rmse(X_train_std, k=k)})
k_df = pd.DataFrame(k_rows).sort_values('rmse').reset_index(drop=True)
best_k = int(k_df.iloc[0]['k'])

imputer = KNNImputer(n_neighbors=best_k)
X_train_final = pd.DataFrame(imputer.fit_transform(X_train_std), columns=X_train_std.columns, index=X_train_std.index)
X_cal_final = pd.DataFrame(imputer.transform(X_cal_std), columns=X_cal_std.columns, index=X_cal_std.index)
X_test_final = pd.DataFrame(imputer.transform(X_test_std), columns=X_test_std.columns, index=X_test_std.index)

clip_lower = X_train_final.quantile(0.01)
clip_upper = X_train_final.quantile(0.99)
X_train_final = X_train_final.clip(lower=clip_lower, upper=clip_upper, axis=1)
X_cal_final = X_cal_final.clip(lower=clip_lower, upper=clip_upper, axis=1)
X_test_final = X_test_final.clip(lower=clip_lower, upper=clip_upper, axis=1)

prep_bundle = {
    'scaler': scaler,
    'imputer': imputer,
    'train_medians': train_medians,
    'feature_columns': X_train_final.columns.tolist(),
    'clip_lower': clip_lower.to_dict(),
    'clip_upper': clip_upper.to_dict(),
    'best_k': best_k,
    'collinearity_threshold': COLLINEARITY_THRESHOLD,
}
joblib.dump(prep_bundle, ARTIFACT_DIR / 'preprocessing_bundle.joblib')
k_df.to_csv(ARTIFACT_DIR / 'knn_imputer_search.csv', index=False)

print(f'Encoded features: {X.shape[1]}')
print(f'Train/Cal/Test shapes: {X_train_final.shape}, {X_cal_final.shape}, {X_test_final.shape}')
print(f'Best KNNImputer k: {best_k}')
print(k_df.to_string(index=False))

Encoded features: 10
Train/Cal/Test shapes: (236022, 10), (50576, 10), (50577, 10)
Best KNNImputer k: 11
 k     rmse
11 1.843200
 9 2.017555
 5 2.531847
 7 2.579276
 3 2.851012


In [ ]:
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold


def safe_predict_proba(model, X_df: pd.DataFrame) -> np.ndarray:
    if hasattr(model, 'predict_proba'):
        p = model.predict_proba(X_df)
        return p[:, 1] if p.ndim == 2 else p
    if hasattr(model, 'decision_function'):
        d = np.asarray(model.decision_function(X_df), dtype=float)
        return 1.0 / (1.0 + np.exp(-d))
    return np.asarray(model.predict(X_df), dtype=float)


def metric_pack(y_true, y_prob, threshold: float = 0.5):
    y_pred = (np.asarray(y_prob) >= threshold).astype(int)
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'auc': roc_auc_score(y_true, y_prob),
        'logloss': log_loss(y_true, np.clip(y_prob, 1e-6, 1 - 1e-6)),
    }


def selection_score(metrics: dict) -> float:
    return 0.50 * metrics['f1'] + 0.30 * metrics['auc'] + 0.20 * metrics['recall']


def sample_param_grid(grid_dict: dict, max_trials: int, seed: int):
    full_grid = list(ParameterGrid(grid_dict))
    if len(full_grid) <= max_trials:
        return full_grid
    rng = np.random.default_rng(seed)
    picks = rng.choice(len(full_grid), size=max_trials, replace=False)
    return [full_grid[i] for i in sorted(picks)]


X_fit, X_valid, y_fit, y_valid = train_test_split(
    X_train_final, y_train, test_size=0.15, random_state=RANDOM_SEED, stratify=y_train
)

# More rigorous GPU search spaces; bounded with deterministic sampling.
gpu_search_spaces = {
    'RandomForest_GPU_XGBRF': {
        'model_cls': XGBRFClassifier,
        'base_params': {
            'objective': 'binary:logistic',
            'eval_metric': 'logloss',
            'tree_method': 'hist',
            'device': 'cuda',
            'random_state': RANDOM_SEED,
            'n_jobs': -1,
        },
        'grid': sample_param_grid({
            'n_estimators': [500, 800, 1200],
            'max_depth': [8, 10, 12],
            'subsample': [0.7, 0.85, 1.0],
            'colsample_bynode': [0.7, 0.85, 1.0],
            'reg_lambda': [1.0, 2.0],
        }, max_trials=72, seed=RANDOM_SEED + 1),
    },
    'XGBoost_GPU': {
        'model_cls': XGBClassifier,
        'base_params': {
            'objective': 'binary:logistic',
            'eval_metric': 'logloss',
            'tree_method': 'hist',
            'device': 'cuda',
            'random_state': RANDOM_SEED,
            'n_jobs': -1,
            'verbosity': 0,
        },
        'grid': sample_param_grid({
            'n_estimators': [500, 900, 1300],
            'max_depth': [5, 7, 9],
            'learning_rate': [0.02, 0.04, 0.06],
            'subsample': [0.75, 0.9],
            'colsample_bytree': [0.75, 0.9],
            'min_child_weight': [1, 3, 5],
            'gamma': [0.0, 0.2],
            'reg_lambda': [1.0, 2.0],
        }, max_trials=96, seed=RANDOM_SEED + 2),
    },
    'CatBoost_GPU': {
        'model_cls': CatBoostClassifier,
        'base_params': {
            'loss_function': 'Logloss',
            'eval_metric': 'AUC',
            'task_type': 'GPU',
            'devices': '0',
            'random_seed': RANDOM_SEED,
            'thread_count': -1,
            'verbose': False,
        },
        'grid': sample_param_grid({
            'iterations': [600, 1000, 1400],
            'depth': [6, 8, 10],
            'learning_rate': [0.02, 0.04, 0.06],
            'l2_leaf_reg': [3, 7, 11],
            'bagging_temperature': [0.0, 1.0],
        }, max_trials=96, seed=RANDOM_SEED + 3),
    },
    'LightGBM_GPU': {
        'model_cls': LGBMClassifier,
        'base_params': {
            'objective': 'binary',
            'device_type': 'gpu',
            'random_state': RANDOM_SEED,
            'n_jobs': -1,
            'verbosity': -1,
        },
        'grid': sample_param_grid({
            'n_estimators': [500, 900, 1300],
            'learning_rate': [0.02, 0.04, 0.06],
            'num_leaves': [31, 63, 127],
            'subsample': [0.75, 0.9],
            'colsample_bytree': [0.75, 0.9],
            'min_child_samples': [20, 40, 80],
            'reg_lambda': [0.0, 1.0],
        }, max_trials=96, seed=RANDOM_SEED + 4),
    },
}

# CPU search remains lighter but still expanded a bit.
cpu_search_spaces = {
    'LogReg_CPU': {
        'model_cls': LogisticRegression,
        'base_params': {
            'max_iter': 5000,
            'class_weight': 'balanced',
            'solver': 'liblinear',
            'n_jobs': -1,
            'random_state': RANDOM_SEED,
        },
        'grid': list(ParameterGrid({'C': [0.25, 0.5, 1.0, 2.0, 4.0]})),
    },
    'AdaBoost_CPU': {
        'model_cls': AdaBoostClassifier,
        'base_params': {
            'random_state': RANDOM_SEED,
        },
        'grid': list(ParameterGrid({
            'n_estimators': [150, 300, 500],
            'learning_rate': [0.02, 0.05, 0.1],
        })),
    },
    'KNN_CPU': {
        'model_cls': KNeighborsClassifier,
        'base_params': {
            'weights': 'distance',
            'n_jobs': -1,
        },
        'grid': list(ParameterGrid({'n_neighbors': sorted(set([best_k, max(3, best_k - 2), best_k + 2, best_k + 4]))})),
    },
    'GaussianNB_CPU': {
        'model_cls': GaussianNB,
        'base_params': {},
        'grid': list(ParameterGrid({'var_smoothing': [1e-10, 1e-9, 1e-8, 1e-7]})),
    },
}


def cv_rescore(model_cls, params: dict, X_df: pd.DataFrame, y_sr: pd.Series, splits: int = 3):
    skf = StratifiedKFold(n_splits=splits, shuffle=True, random_state=RANDOM_SEED)
    fold_metrics = []
    for fold_id, (tr_idx, va_idx) in enumerate(skf.split(X_df, y_sr), start=1):
        X_tr = X_df.iloc[tr_idx]
        y_tr = y_sr.iloc[tr_idx]
        X_va = X_df.iloc[va_idx]
        y_va = y_sr.iloc[va_idx]

        fold_model = model_cls(**params)
        fold_model.fit(X_tr, y_tr)
        p_va = np.clip(safe_predict_proba(fold_model, X_va), 1e-6, 1 - 1e-6)
        m = metric_pack(y_va, p_va)
        fold_metrics.append(m)

    avg_metrics = {
        'accuracy': float(np.mean([m['accuracy'] for m in fold_metrics])),
        'precision': float(np.mean([m['precision'] for m in fold_metrics])),
        'recall': float(np.mean([m['recall'] for m in fold_metrics])),
        'f1': float(np.mean([m['f1'] for m in fold_metrics])),
        'auc': float(np.mean([m['auc'] for m in fold_metrics])),
        'logloss': float(np.mean([m['logloss'] for m in fold_metrics])),
    }
    return avg_metrics


def tune_and_fit(search_spaces: dict, run_label: str, enable: bool = True):
    rows = []
    trained = {}
    if not enable:
        print(f'{run_label} skipped because runtime requirements are unavailable.')
        return rows, trained

    for model_name, spec in search_spaces.items():
        print(f'\n=== {run_label}: {model_name} ===')
        quick_rows = []
        best_error = None

        # Stage 1: quick screen on fit/valid split.
        for params in spec['grid']:
            full_params = {**spec['base_params'], **params}
            t0 = time.time()
            try:
                model = spec['model_cls'](**full_params)
                model.fit(X_fit, y_fit)
                p_valid = np.clip(safe_predict_proba(model, X_valid), 1e-6, 1 - 1e-6)
                metrics = metric_pack(y_valid, p_valid)
                quick_rows.append({
                    'params': full_params,
                    'quick_score': selection_score(metrics),
                    'quick_metrics': metrics,
                    'quick_seconds': round(time.time() - t0, 2),
                })
            except Exception as exc:
                best_error = f'{type(exc).__name__}: {exc}'

        if not quick_rows:
            rows.append({
                'model_name': model_name,
                'run_group': run_label,
                'status': 'failed',
                'error': best_error,
            })
            print(f'Failed: {best_error}')
            continue

        quick_df = pd.DataFrame(quick_rows).sort_values('quick_score', ascending=False).reset_index(drop=True)
        shortlist_n = 8 if run_label == 'gpu' else 4
        shortlist = quick_df.head(min(shortlist_n, len(quick_df)))

        # Stage 2: 3-fold CV on top candidates.
        cv_rows = []
        for _, cand in shortlist.iterrows():
            params = cand['params']
            try:
                cv_metrics = cv_rescore(spec['model_cls'], params, X_train_final, y_train, splits=3)
                cv_rows.append({
                    'params': params,
                    'cv_score': selection_score(cv_metrics),
                    'cv_metrics': cv_metrics,
                    'quick_score': float(cand['quick_score']),
                })
            except Exception as exc:
                best_error = f'{type(exc).__name__}: {exc}'

        if not cv_rows:
            rows.append({
                'model_name': model_name,
                'run_group': run_label,
                'status': 'failed',
                'error': best_error,
            })
            print(f'Failed during CV stage: {best_error}')
            continue

        cv_df = pd.DataFrame(cv_rows).sort_values(['cv_score', 'quick_score'], ascending=False).reset_index(drop=True)
        best_params = cv_df.iloc[0]['params']
        best_cv_metrics = cv_df.iloc[0]['cv_metrics']

        final_model = spec['model_cls'](**best_params)
        train_start = time.time()
        final_model.fit(X_train_final, y_train)
        train_seconds = round(time.time() - train_start, 2)

        model_path = MODEL_DIR / f'{model_name}.joblib'
        joblib.dump(final_model, model_path)
        p_cal = np.clip(safe_predict_proba(final_model, X_cal_final), 1e-6, 1 - 1e-6)
        cal_metrics = metric_pack(y_cal, p_cal)
        trained[model_name] = final_model

        rows.append({
            'model_name': model_name,
            'run_group': run_label,
            'status': 'ok',
            'model_path': str(model_path),
            'search_space_size': len(spec['grid']),
            'shortlist_size': int(len(shortlist)),
            'best_params': json.dumps(best_params, default=str),
            'cv_score': float(cv_df.iloc[0]['cv_score']),
            'cv_f1': best_cv_metrics['f1'],
            'cv_auc': best_cv_metrics['auc'],
            'cv_recall': best_cv_metrics['recall'],
            'train_seconds': train_seconds,
            'cal_accuracy': cal_metrics['accuracy'],
            'cal_precision': cal_metrics['precision'],
            'cal_recall': cal_metrics['recall'],
            'cal_f1': cal_metrics['f1'],
            'cal_auc': cal_metrics['auc'],
            'cal_logloss': cal_metrics['logloss'],
        })
        print({
            'search_space_size': len(spec['grid']),
            'shortlist_size': int(len(shortlist)),
            'cv_score': round(float(cv_df.iloc[0]['cv_score']), 6),
            'cv_f1': round(best_cv_metrics['f1'], 6),
            'cal_f1': round(cal_metrics['f1'], 6),
            'cal_auc': round(cal_metrics['auc'], 6),
        })

    return rows, trained


gpu_rows, gpu_models = tune_and_fit(gpu_search_spaces, 'gpu', enable=GPU_AVAILABLE)
cpu_rows, cpu_models = tune_and_fit(cpu_search_spaces, 'cpu', enable=True)

results_df = pd.DataFrame(gpu_rows + cpu_rows)
if not results_df.empty:
    results_df = results_df.sort_values(['status', 'cal_f1', 'cal_auc'], ascending=[True, False, False]).reset_index(drop=True)
results_path = ARTIFACT_DIR / 'model_selection_results_clinical_dietary_anthro.csv'
results_df.to_csv(results_path, index=False)
print(results_df.to_string(index=False))
print(f'Saved model selection results: {results_path}')


=== gpu: RandomForest_GPU_XGBRF ===
{'best_params': {'objective': 'binary:logistic', 'eval_metric': 'logloss', 'tree_method': 'hist', 'device': 'cuda', 'random_state': 2026, 'n_jobs': -1, 'colsample_bynode': 1.0, 'max_depth': 10, 'n_estimators': 400, 'subsample': 0.8}, 'tune_score': 0.542415, 'cal_f1': 0.477206, 'cal_auc': 0.746369}

=== gpu: XGBoost_GPU ===
{'best_params': {'objective': 'binary:logistic', 'eval_metric': 'logloss', 'tree_method': 'hist', 'device': 'cuda', 'random_state': 2026, 'n_jobs': -1, 'verbosity': 0, 'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 6, 'min_child_weight': 1, 'n_estimators': 800, 'subsample': 0.8}, 'tune_score': 0.525438, 'cal_f1': 0.46305, 'cal_auc': 0.748187}

=== gpu: CatBoost_GPU ===


Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric perio

{'best_params': {'loss_function': 'Logloss', 'eval_metric': 'AUC', 'task_type': 'GPU', 'devices': '0', 'random_seed': 2026, 'thread_count': -1, 'verbose': False, 'depth': 8, 'iterations': 500, 'l2_leaf_reg': 7, 'learning_rate': 0.05}, 'tune_score': 0.525996, 'cal_f1': 0.460631, 'cal_auc': 0.74992}

=== gpu: LightGBM_GPU ===


1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


{'best_params': {'objective': 'binary', 'device_type': 'gpu', 'random_state': 2026, 'n_jobs': -1, 'verbosity': -1, 'colsample_bytree': 0.8, 'learning_rate': 0.05, 'min_child_samples': 40, 'n_estimators': 400, 'num_leaves': 63, 'subsample': 1.0}, 'tune_score': 0.530004, 'cal_f1': 0.463366, 'cal_auc': 0.749537}

=== cpu: LogReg_CPU ===
{'best_params': {'max_iter': 5000, 'class_weight': 'balanced', 'solver': 'liblinear', 'n_jobs': -1, 'random_state': 2026, 'C': 2.0}, 'tune_score': 0.639379, 'cal_f1': 0.574976, 'cal_auc': 0.738237}

=== cpu: AdaBoost_CPU ===
{'best_params': {'random_state': 2026, 'learning_rate': 0.05, 'n_estimators': 300}, 'tune_score': 0.428854, 'cal_f1': 0.346198, 'cal_auc': 0.732806}

=== cpu: KNN_CPU ===
{'best_params': {'weights': 'distance', 'n_jobs': -1, 'n_neighbors': 9}, 'tune_score': 0.511485, 'cal_f1': 0.449204, 'cal_auc': 0.693323}

=== cpu: GaussianNB_CPU ===
{'best_params': {'var_smoothing': 1e-09}, 'tune_score': 0.528497, 'cal_f1': 0.473091, 'cal_auc': 0.72

In [10]:
def expected_calibration_error(y_true, y_prob, n_bins: int = 15) -> float:
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    idx = np.digitize(y_prob, bins) - 1
    ece = 0.0
    for i in range(n_bins):
        mask = idx == i
        if np.any(mask):
            conf = y_prob[mask].mean()
            acc = y_true[mask].mean()
            ece += mask.mean() * abs(acc - conf)
    return float(ece)

ok_results = results_df[results_df['status'] == 'ok'].copy() if not results_df.empty else pd.DataFrame()
if ok_results.empty:
    raise RuntimeError('No trained models succeeded, so calibration cannot proceed.')

top_models = ok_results.sort_values(['cal_f1', 'cal_auc'], ascending=False).head(min(6, len(ok_results))).copy()
X_cal_fit, X_cal_eval, y_cal_fit, y_cal_eval = train_test_split(
    X_cal_final, y_cal, test_size=0.5, random_state=RANDOM_SEED, stratify=y_cal
)

venn_available = True
try:
    from venn_abers import VennAbers
except Exception:
    venn_available = False

calib_rows = []
for _, row in top_models.iterrows():
    model = joblib.load(row['model_path'])
    model_name = row['model_name']
    p_fit = np.clip(safe_predict_proba(model, X_cal_fit), 1e-6, 1 - 1e-6)
    p_eval = np.clip(safe_predict_proba(model, X_cal_eval), 1e-6, 1 - 1e-6)

    base_pred = (p_eval >= 0.5).astype(int)
    calib_rows.append({
        'model_name': model_name,
        'method': 'none',
        'cal_logloss': log_loss(y_cal_eval, p_eval),
        'cal_ece': expected_calibration_error(y_cal_eval, p_eval),
        'cal_auc': roc_auc_score(y_cal_eval, p_eval),
        'cal_f1': f1_score(y_cal_eval, base_pred, zero_division=0),
        'model_path': row['model_path'],
    })

    platt = LogisticRegression(max_iter=3000, solver='liblinear', random_state=RANDOM_SEED)
    platt.fit(p_fit.reshape(-1, 1), y_cal_fit.values)
    p_sig = np.clip(platt.predict_proba(p_eval.reshape(-1, 1))[:, 1], 1e-6, 1 - 1e-6)
    sig_pred = (p_sig >= 0.5).astype(int)
    calib_rows.append({
        'model_name': model_name,
        'method': 'sigmoid',
        'cal_logloss': log_loss(y_cal_eval, p_sig),
        'cal_ece': expected_calibration_error(y_cal_eval, p_sig),
        'cal_auc': roc_auc_score(y_cal_eval, p_sig),
        'cal_f1': f1_score(y_cal_eval, sig_pred, zero_division=0),
        'model_path': row['model_path'],
    })

    iso = IsotonicRegression(out_of_bounds='clip')
    iso.fit(p_fit, y_cal_fit.values)
    p_iso = np.clip(iso.predict(p_eval), 1e-6, 1 - 1e-6)
    iso_pred = (p_iso >= 0.5).astype(int)
    calib_rows.append({
        'model_name': model_name,
        'method': 'isotonic',
        'cal_logloss': log_loss(y_cal_eval, p_iso),
        'cal_ece': expected_calibration_error(y_cal_eval, p_iso),
        'cal_auc': roc_auc_score(y_cal_eval, p_iso),
        'cal_f1': f1_score(y_cal_eval, iso_pred, zero_division=0),
        'model_path': row['model_path'],
    })

    if venn_available:
        va = VennAbers()
        va.fit(np.column_stack([1.0 - p_fit, p_fit]), y_cal_fit.values)
        _, p_va_raw = va.predict_proba(np.column_stack([1.0 - p_eval, p_eval]))
        p_va_raw = np.asarray(p_va_raw)
        p_va = p_va_raw[:, 1] if p_va_raw.ndim == 2 else p_va_raw.reshape(-1)
        p_va = np.clip(p_va, 1e-6, 1 - 1e-6)
        va_pred = (p_va >= 0.5).astype(int)
        calib_rows.append({
            'model_name': model_name,
            'method': 'venn_abers',
            'cal_logloss': log_loss(y_cal_eval, p_va),
            'cal_ece': expected_calibration_error(y_cal_eval, p_va),
            'cal_auc': roc_auc_score(y_cal_eval, p_va),
            'cal_f1': f1_score(y_cal_eval, va_pred, zero_division=0),
            'model_path': row['model_path'],
        })

calibration_df = pd.DataFrame(calib_rows)
calibration_df['logloss_rank'] = calibration_df['cal_logloss'].rank(method='min')
calibration_df['ece_rank'] = calibration_df['cal_ece'].rank(method='min')
calibration_df['avg_rank'] = (calibration_df['logloss_rank'] + calibration_df['ece_rank']) / 2.0
best_calib = calibration_df.sort_values(['avg_rank', 'cal_logloss', 'cal_ece']).iloc[0]

calibration_path = ARTIFACT_DIR / 'calibration_results_clinical_dietary_anthro.csv'
calibration_df.to_csv(calibration_path, index=False)
print(calibration_df.sort_values(['avg_rank', 'cal_logloss', 'cal_ece']).head(12).to_string(index=False))
print(f"Best calibration choice: {best_calib['model_name']} + {best_calib['method']}")
print(f'Saved calibration results: {calibration_path}')

            model_name     method  cal_logloss  cal_ece  cal_auc   cal_f1                                                                                                       model_path  logloss_rank  ece_rank  avg_rank
          CatBoost_GPU   isotonic     0.539308 0.006228 0.749969 0.519517           /workspace/Thesis-part-2/V2.2.1.1/clinical_dietary_anthro_gpu_cpu_artifacts/models/CatBoost_GPU.joblib           4.0       2.0       3.0
          CatBoost_GPU       none     0.539073 0.006961 0.750526 0.460819           /workspace/Thesis-part-2/V2.2.1.1/clinical_dietary_anthro_gpu_cpu_artifacts/models/CatBoost_GPU.joblib           2.0       5.0       3.5
          CatBoost_GPU venn_abers     0.539294 0.006692 0.750004 0.519484           /workspace/Thesis-part-2/V2.2.1.1/clinical_dietary_anthro_gpu_cpu_artifacts/models/CatBoost_GPU.joblib           3.0       4.0       3.5
          LightGBM_GPU       none     0.538923 0.007741 0.750273 0.464828           /workspace/Thesis-part-2/V2.2.1.

In [11]:
best_model = joblib.load(best_calib['model_path'])
best_method = best_calib['method']
p_cal_base = np.clip(safe_predict_proba(best_model, X_cal_final), 1e-6, 1 - 1e-6)
p_test_base = np.clip(safe_predict_proba(best_model, X_test_final), 1e-6, 1 - 1e-6)

if best_method == 'sigmoid':
    final_calibrator = LogisticRegression(max_iter=3000, solver='liblinear', random_state=RANDOM_SEED)
    final_calibrator.fit(p_cal_base.reshape(-1, 1), y_cal.values)
    p_test_final = np.clip(final_calibrator.predict_proba(p_test_base.reshape(-1, 1))[:, 1], 1e-6, 1 - 1e-6)
elif best_method == 'isotonic':
    final_calibrator = IsotonicRegression(out_of_bounds='clip')
    final_calibrator.fit(p_cal_base, y_cal.values)
    p_test_final = np.clip(final_calibrator.predict(p_test_base), 1e-6, 1 - 1e-6)
elif best_method == 'venn_abers':
    if not venn_available:
        raise RuntimeError('Best method is venn_abers but package is unavailable.')
    va = VennAbers()
    va.fit(np.column_stack([1.0 - p_cal_base, p_cal_base]), y_cal.values)
    _, p_va_raw = va.predict_proba(np.column_stack([1.0 - p_test_base, p_test_base]))
    p_va_raw = np.asarray(p_va_raw)
    p_test_final = np.clip(p_va_raw[:, 1] if p_va_raw.ndim == 2 else p_va_raw.reshape(-1), 1e-6, 1 - 1e-6)
else:
    p_test_final = p_test_base

test_metrics = metric_pack(y_test, p_test_final, threshold=0.5)
test_metrics_row = {
    'best_model': best_calib['model_name'],
    'calibration_method': best_method,
    'test_accuracy': test_metrics['accuracy'],
    'test_precision': test_metrics['precision'],
    'test_recall': test_metrics['recall'],
    'test_f1': test_metrics['f1'],
    'test_auc': test_metrics['auc'],
    'test_logloss': test_metrics['logloss'],
    'best_k_imputer': best_k,
    'collinearity_threshold': COLLINEARITY_THRESHOLD,
}

test_metrics_df = pd.DataFrame([test_metrics_row])
test_metrics_path = ARTIFACT_DIR / 'best_calibrated_test_metrics.csv'
test_metrics_df.to_csv(test_metrics_path, index=False)

config_path = ARTIFACT_DIR / 'best_model_config.json'
with open(config_path, 'w', encoding='utf-8') as f:
    json.dump(test_metrics_row, f, indent=2)

print(test_metrics_df.to_string(index=False))
print(f'Saved: {test_metrics_path}')
print(f'Saved: {config_path}')
print('\nGenerated artifacts:')
for path in sorted(ARTIFACT_DIR.glob('*')):
    if path.is_file():
        print(f'- {path.name}')
print('\nModel files:')
for path in sorted(MODEL_DIR.glob('*.joblib')):
    print(f'- {path.name}')

  best_model calibration_method  test_accuracy  test_precision  test_recall  test_f1  test_auc  test_logloss  best_k_imputer  collinearity_threshold
CatBoost_GPU           isotonic       0.701979        0.586589     0.358113 0.444723  0.747192      0.542383              11                     0.7
Saved: /workspace/Thesis-part-2/V2.2.1.1/clinical_dietary_anthro_gpu_cpu_artifacts/best_calibrated_test_metrics.csv
Saved: /workspace/Thesis-part-2/V2.2.1.1/clinical_dietary_anthro_gpu_cpu_artifacts/best_model_config.json

Generated artifacts:
- best_calibrated_test_metrics.csv
- best_model_config.json
- calibration_results_clinical_dietary_anthro.csv
- collinearity_pairs.csv
- collinearity_report.json
- knn_imputer_search.csv
- manual_drop_report.csv
- merged_clinical_dietary_anthro_multiyear.csv
- model_selection_results_clinical_dietary_anthro.csv
- preprocessing_bundle.joblib

Model files:
- AdaBoost_CPU.joblib
- CatBoost_GPU.joblib
- GaussianNB_CPU.joblib
- KNN_CPU.joblib
- LightGBM_GPU.j